# Experiment C - Multi-channel xT Feature Clustering

## 1. Markdown Introduction

This notebook performs Experiment C for unsupervised clustering of team-season tactical styles using a multi-channel team-season xT feature dataset.

Each team-season is represented by several 16 by 12 spatial matrices. The channels may include created positive xT by start zone, received positive xT by end zone, pass-created positive xT by start zone, carry-created positive xT by start zone, and action-count distribution by start zone.

The goal is to cluster team-seasons using richer possession threat-creation information than a single xT created matrix. This representation may capture more tactical nuance than Experiment A or Experiment B, but it also has more features and may require stronger dimensionality reduction.

Experiment C should be compared with Experiment A and Experiment B when their outputs exist. Do not overclaim tactical labels: xT-based features mainly describe possession threat creation and ball progression. They do not directly measure pressing intensity, defensive block height, or counterpressing.


## 2. Imports and Configuration

This setup cell defines the Experiment C contract: one processed multi-channel feature CSV, one output folder, a 16 by 12 pitch grid, PCA variance retention, and KMeans settings. The notebook does not retrain xT, parse raw StatsBomb files, recalculate action-level xT, rebuild team-season matrices, or use PNG heatmaps as machine learning input.


In [ ]:
from pathlib import Path
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import normalized_mutual_info_score
from sklearn.metrics import pairwise_distances_argmin_min


INPUT_FILE = Path("outputs/team_season_features_multichannel_xt_distribution.csv")
OUTPUT_DIR = Path("outputs/experiment_C_multichannel/")
EXPERIMENT_A_CLUSTERED_FILE = Path("outputs/experiment_A_raw/clustered_team_seasons_raw.csv")
EXPERIMENT_B_CLUSTERED_FILE = Path("outputs/experiment_B_smoothed/clustered_team_seasons_smoothed.csv")

EXPERIMENT_NAME = "Experiment C - Multi-channel xT Feature Clustering"

GRID_L = 16
GRID_W = 12
N_ZONES = GRID_L * GRID_W

MIN_MATCH_COUNT = 5
K_RANGE = range(3, 11)
FINAL_K = 5
RANDOM_STATE = 42
PCA_VARIANCE_TARGET = 0.85

CHANNEL_PREFIXES = [
    "created_z",
    "received_z",
    "pass_created_z",
    "carry_created_z",
    "action_count_z",
]

CHANNEL_OUTPUT_NAMES = {
    "created_z": "created",
    "received_z": "received",
    "pass_created_z": "pass_created",
    "carry_created_z": "carry_created",
    "action_count_z": "action_count",
}

CHANNEL_DISPLAY_NAMES = {
    "created_z": "Created xT",
    "received_z": "Received xT",
    "pass_created_z": "Pass-created xT",
    "carry_created_z": "Carry-created xT",
    "action_count_z": "Action count",
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Input file: {INPUT_FILE}")
print(f"Output folder: {OUTPUT_DIR}")


## 3. Load and Validate Data

This section loads only the processed multi-channel feature CSV. It detects available spatial channels, requires each included channel to contain exactly 192 zone columns, and keeps only valid detected channels for clustering.


In [ ]:
if not INPUT_FILE.exists():
    nearby_files = sorted(str(path) for path in INPUT_FILE.parent.glob("*team_season*"))
    nearby_preview = "\n".join(nearby_files[:30]) if nearby_files else "No nearby team-season files found."
    raise FileNotFoundError(
        f"Expected multi-channel xT feature file not found: {INPUT_FILE}\n\n"
        "Experiment C should load a processed CSV with one or more complete 192-zone channels using prefixes: "
        f"{CHANNEL_PREFIXES}.\n"
        "Create or copy that processed file before running this notebook.\n\n"
        f"Nearby team-season files found:\n{nearby_preview}"
    )

data = pd.read_csv(INPUT_FILE)

print(f"Dataset shape: {data.shape}")
print(f"Number of rows: {len(data):,}")
print(f"Number of columns: {data.shape[1]:,}")

if "team_name" in data.columns:
    print(f"Unique teams: {data['team_name'].nunique(dropna=True):,}")
elif "team_id" in data.columns:
    print(f"Unique teams: {data['team_id'].nunique(dropna=True):,}")

if "season_name" in data.columns:
    print(f"Seasons: {data['season_name'].nunique(dropna=True):,}")
elif "season_id" in data.columns:
    print(f"Seasons: {data['season_id'].nunique(dropna=True):,}")

if "competition_name" in data.columns:
    print(f"Competitions: {data['competition_name'].nunique(dropna=True):,}")
elif "competition_id" in data.columns:
    print(f"Competitions: {data['competition_id'].nunique(dropna=True):,}")


def zone_number_from_column(column_name, prefix):
    match = re.search(rf"^{re.escape(prefix)}(\d+)$", column_name)
    if match is None:
        return None
    return int(match.group(1))


detected_channels = {}
channel_detection_rows = []

for prefix in CHANNEL_PREFIXES:
    pairs = []
    for column in data.columns:
        zone = zone_number_from_column(column, prefix)
        if zone is not None:
            pairs.append((zone, column))

    pairs = sorted(pairs)
    zones = [zone for zone, _ in pairs]
    columns = [column for _, column in pairs]

    if len(columns) == N_ZONES and zones == list(range(N_ZONES)):
        detected_channels[prefix] = columns
        status = "included"
    elif len(columns) == 0:
        status = "missing"
    else:
        status = "skipped_incomplete"
        warnings.warn(
            f"Channel {prefix!r} has {len(columns)} columns, not a complete z000-z191 set. "
            "Skipping this channel."
        )

    channel_detection_rows.append(
        {
            "channel_prefix": prefix,
            "n_columns_found": len(columns),
            "status": status,
        }
    )

if not detected_channels:
    raise ValueError("No valid complete 192-zone multi-channel feature set was found.")

channel_detection = pd.DataFrame(channel_detection_rows)
feature_columns = [column for prefix in CHANNEL_PREFIXES for column in detected_channels.get(prefix, [])]
metadata_columns = [column for column in data.columns if column not in feature_columns]

feature_numeric = data[feature_columns].apply(pd.to_numeric, errors="coerce")
coerced_to_missing = feature_numeric.isna() & data[feature_columns].notna()
non_numeric_count = int(coerced_to_missing.sum().sum())
if non_numeric_count:
    raise ValueError(f"Found {non_numeric_count:,} non-numeric values in selected feature columns.")
data[feature_columns] = feature_numeric

all_missing_features = data[feature_columns].columns[data[feature_columns].isna().all()].tolist()
if all_missing_features:
    raise ValueError(f"These selected feature columns are entirely missing: {all_missing_features[:10]}")

id_duplicate_keys = ["competition_id", "season_id", "team_id"]
name_duplicate_keys = ["competition_name", "season_name", "team_name"]
if all(column in data.columns for column in id_duplicate_keys):
    duplicate_key_columns = id_duplicate_keys
elif all(column in data.columns for column in name_duplicate_keys):
    duplicate_key_columns = name_duplicate_keys
else:
    duplicate_key_columns = []

if duplicate_key_columns:
    duplicate_rows = data.duplicated(subset=duplicate_key_columns, keep=False)
    print(f"Duplicate team-season rows using {duplicate_key_columns}: {int(duplicate_rows.sum()):,}")
else:
    print("No complete team-season key was available for duplicate-row checking.")

metadata_missing = data[metadata_columns].isna().sum().sort_values(ascending=False)
metadata_missing = metadata_missing[metadata_missing > 0]
feature_missing_counts = data[feature_columns].isna().sum()

print("Detected channels:")
display(channel_detection)
print(f"Channels used: {list(detected_channels.keys())}")
print(f"Total number of feature columns used: {len(feature_columns):,}")
print(f"Metadata columns: {metadata_columns}")
print(f"Missing values in metadata columns: {int(metadata_missing.sum()):,}")
if not metadata_missing.empty:
    display(metadata_missing.head(20).rename("missing_values").to_frame())
print(f"Missing values in selected feature columns: {int(feature_missing_counts.sum()):,}")

display(data[metadata_columns + feature_columns[:5]].head())


## 4. Data Filtering

This section removes low-quality samples before clustering. Rows are removed when they have too few matches, no multi-channel feature signal, or missing selected feature values. The filtering report records the sample definition.


In [ ]:
initial_row_count = len(data)
filtered = data.copy()

removed_low_match_count = 0
removed_zero_feature_sum = 0
removed_missing_features = 0

if "match_count" in filtered.columns:
    filtered["match_count"] = pd.to_numeric(filtered["match_count"], errors="coerce")
    low_match_mask = filtered["match_count"].fillna(-np.inf) < MIN_MATCH_COUNT
    removed_low_match_count = int(low_match_mask.sum())
    filtered = filtered.loc[~low_match_mask].copy()
else:
    print("Column 'match_count' not found, so the minimum-match filter is skipped.")

total_feature_sum = filtered[feature_columns].sum(axis=1, skipna=False)
zero_feature_mask = np.isclose(total_feature_sum.fillna(np.nan), 0)
removed_zero_feature_sum = int(zero_feature_mask.sum())
filtered = filtered.loc[~zero_feature_mask].copy()

missing_feature_mask = filtered[feature_columns].isna().any(axis=1)
removed_missing_features = int(missing_feature_mask.sum())
filtered = filtered.loc[~missing_feature_mask].copy()

final_row_count = len(filtered)

filtering_report = pd.DataFrame(
    [
        {"step": "initial_rows", "row_count": initial_row_count},
        {"step": "removed_low_match_count", "row_count": removed_low_match_count},
        {"step": "removed_zero_total_feature_sum", "row_count": removed_zero_feature_sum},
        {"step": "removed_missing_feature_values", "row_count": removed_missing_features},
        {"step": "final_rows", "row_count": final_row_count},
    ]
)
filtering_report.to_csv(OUTPUT_DIR / "filtering_report_multichannel.csv", index=False)

print(f"Initial row count: {initial_row_count:,}")
print(f"Rows removed because of low match_count: {removed_low_match_count:,}")
print(f"Rows removed because total feature sum is zero: {removed_zero_feature_sum:,}")
print(f"Rows removed because of missing feature values: {removed_missing_features:,}")
print(f"Final row count: {final_row_count:,}")
print(f"Saved filtering report: {OUTPUT_DIR / 'filtering_report_multichannel.csv'}")
display(filtering_report)

if final_row_count <= FINAL_K:
    raise ValueError(
        f"Not enough filtered samples ({final_row_count}) for FINAL_K={FINAL_K}. "
        "Lower FINAL_K or relax filtering."
    )


## 5. Feature Matrix Preparation

All detected multi-channel feature columns are used for clustering. Each channel is expected to be a distribution over pitch zones. Keeping the channels separate preserves their different tactical meanings instead of merging everything into one xT map.


In [ ]:
metadata_filtered = filtered[metadata_columns].reset_index(drop=True)
X = filtered[feature_columns].reset_index(drop=True)

channel_row_sum_rows = []
for prefix, columns in detected_channels.items():
    row_sums = filtered[columns].sum(axis=1)
    nonzero_row_sums = row_sums.loc[~np.isclose(row_sums, 0)]
    max_abs_deviation_from_one = (
        float((nonzero_row_sums - 1).abs().max()) if len(nonzero_row_sums) else np.nan
    )
    channel_row_sum_rows.append(
        {
            "channel_prefix": prefix,
            "min_row_sum": row_sums.min(),
            "max_row_sum": row_sums.max(),
            "mean_row_sum": row_sums.mean(),
            "max_abs_deviation_from_one_nonzero": max_abs_deviation_from_one,
        }
    )
    print(
        f"{prefix}: min={row_sums.min():.6f}, max={row_sums.max():.6f}, "
        f"mean={row_sums.mean():.6f}, max |sum - 1|={max_abs_deviation_from_one:.6f}"
    )
    if pd.notna(max_abs_deviation_from_one) and max_abs_deviation_from_one > 0.05:
        print(f"Warning: {prefix} row sums deviate from 1 by more than 0.05.")

channel_row_sum_summary = pd.DataFrame(channel_row_sum_rows)

print(f"Feature matrix shape: {X.shape}")
display(channel_row_sum_summary)


## 6. Standardization

The multi-channel dataset contains different feature groups. StandardScaler prevents one channel or one high-variance zone from dominating KMeans distance calculations.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Scaled feature matrix shape: {X_scaled.shape}")
print(f"Mean of scaled features, average absolute value: {np.abs(X_scaled.mean(axis=0)).mean():.6f}")
print(f"Std of scaled features, average value: {X_scaled.std(axis=0).mean():.6f}")


## 7. PCA Dimensionality Reduction

PCA is especially important for the multi-channel feature set because the number of features is much larger than in Experiment A or B. The retained component count is chosen to explain at least 85 percent of the standardized feature variance by default.


In [ ]:
max_pca_components = min(X_scaled.shape[0], X_scaled.shape[1])
pca_full = PCA(n_components=max_pca_components, random_state=RANDOM_STATE)
X_pca_full = pca_full.fit_transform(X_scaled)

explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

pca_variance_table = pd.DataFrame(
    {
        "component": np.arange(1, len(explained_variance) + 1),
        "explained_variance_ratio": explained_variance,
        "cumulative_explained_variance": cumulative_variance,
    }
)
pca_variance_table.to_csv(OUTPUT_DIR / "pca_explained_variance_multichannel.csv", index=False)

pca_components_used = int(np.searchsorted(cumulative_variance, PCA_VARIANCE_TARGET) + 1)
pca_components_used = min(pca_components_used, max_pca_components)
pca_explained_variance_retained = float(cumulative_variance[pca_components_used - 1])

X_pca = X_pca_full[:, :pca_components_used]
if X_pca_full.shape[1] >= 2:
    X_pca_2d = X_pca_full[:, :2]
else:
    X_pca_2d = np.column_stack([X_pca_full[:, 0], np.zeros(X_pca_full.shape[0])])

print(f"PCA components used: {pca_components_used}")
print(f"Cumulative explained variance retained: {pca_explained_variance_retained:.4f}")
display(pca_variance_table.head(15))

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(
    pca_variance_table["component"],
    pca_variance_table["cumulative_explained_variance"],
    marker="o",
    linewidth=2,
)
ax.axhline(PCA_VARIANCE_TARGET, color="red", linestyle="--", label=f"Target = {PCA_VARIANCE_TARGET:.0%}")
ax.axvline(pca_components_used, color="gray", linestyle=":", label=f"Components used = {pca_components_used}")
ax.set_title("Experiment C PCA Cumulative Explained Variance")
ax.set_xlabel("Number of PCA components")
ax.set_ylabel("Cumulative explained variance")
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "pca_variance_plot_multichannel.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved PCA variance table: {OUTPUT_DIR / 'pca_explained_variance_multichannel.csv'}")
print(f"Saved PCA variance plot: {OUTPUT_DIR / 'pca_variance_plot_multichannel.png'}")


## 8. Evaluate KMeans Cluster Numbers

This section tests k values from 3 through 10 using inertia, silhouette score, Calinski-Harabasz score, and Davies-Bouldin score. Do not choose k only by the highest metric: tactical interpretability, centroid clarity, and representative coherence also matter.


In [ ]:
def fit_kmeans_model(matrix, n_clusters):
    try:
        model = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init="auto")
        model.fit(matrix)
    except TypeError:
        model = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=20)
        model.fit(matrix)
    return model


metric_rows = []
valid_k_values = [k for k in K_RANGE if 1 < k < len(X_pca)]

if not valid_k_values:
    raise ValueError("No valid k values available. Need more filtered samples for clustering evaluation.")

skipped_k_values = [k for k in K_RANGE if k not in valid_k_values]
if skipped_k_values:
    print(f"Skipped k values because there are not enough samples: {skipped_k_values}")

for k in valid_k_values:
    model = fit_kmeans_model(X_pca, k)
    labels = model.labels_
    metric_rows.append(
        {
            "k": k,
            "inertia": model.inertia_,
            "silhouette_score": silhouette_score(X_pca, labels),
            "calinski_harabasz_score": calinski_harabasz_score(X_pca, labels),
            "davies_bouldin_score": davies_bouldin_score(X_pca, labels),
        }
    )

cluster_metric_scores = pd.DataFrame(metric_rows)
cluster_metric_scores.to_csv(OUTPUT_DIR / "cluster_metric_scores_multichannel.csv", index=False)
display(cluster_metric_scores)

fig, axs = plt.subplots(2, 2, figsize=(13, 9))
axs[0, 0].plot(cluster_metric_scores["k"], cluster_metric_scores["inertia"], marker="o")
axs[0, 0].set_title("Inertia by k")
axs[0, 0].set_xlabel("k")
axs[0, 0].set_ylabel("Inertia")

axs[0, 1].plot(cluster_metric_scores["k"], cluster_metric_scores["silhouette_score"], marker="o")
axs[0, 1].set_title("Silhouette score by k")
axs[0, 1].set_xlabel("k")
axs[0, 1].set_ylabel("Silhouette score")

axs[1, 0].plot(cluster_metric_scores["k"], cluster_metric_scores["calinski_harabasz_score"], marker="o")
axs[1, 0].set_title("Calinski-Harabasz score by k")
axs[1, 0].set_xlabel("k")
axs[1, 0].set_ylabel("Calinski-Harabasz score")

axs[1, 1].plot(cluster_metric_scores["k"], cluster_metric_scores["davies_bouldin_score"], marker="o")
axs[1, 1].set_title("Davies-Bouldin score by k")
axs[1, 1].set_xlabel("k")
axs[1, 1].set_ylabel("Davies-Bouldin score")

for ax in axs.flat:
    ax.grid(alpha=0.25)
    ax.axvline(FINAL_K, color="gray", linestyle=":", alpha=0.75)

fig.suptitle("Experiment C - KMeans Cluster Metric Scores", fontsize=16, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "cluster_metric_scores_multichannel.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved cluster metric scores: {OUTPUT_DIR / 'cluster_metric_scores_multichannel.csv'}")
print(f"Saved cluster metric plot: {OUTPUT_DIR / 'cluster_metric_scores_multichannel.png'}")


## 9. Fit Final KMeans Model

The final model uses `FINAL_K = 5` by default and fits KMeans in PCA space. The assigned label is saved as `cluster_multichannel`.


In [ ]:
if FINAL_K >= len(X_pca):
    raise ValueError(f"FINAL_K={FINAL_K} is not valid for {len(X_pca)} samples.")

final_kmeans = fit_kmeans_model(X_pca, FINAL_K)
cluster_multichannel = final_kmeans.labels_

clustered_team_seasons = filtered.reset_index(drop=True).copy()
clustered_team_seasons["cluster_multichannel"] = cluster_multichannel
clustered_team_seasons["pca_1"] = X_pca_2d[:, 0]
clustered_team_seasons["pca_2"] = X_pca_2d[:, 1]

assigned_centroid_indices, assigned_distances = pairwise_distances_argmin_min(X_pca, final_kmeans.cluster_centers_)
clustered_team_seasons["distance_to_centroid"] = assigned_distances

cluster_counts = clustered_team_seasons["cluster_multichannel"].value_counts().sort_index()
cluster_percentages = (cluster_counts / len(clustered_team_seasons) * 100).round(2)
cluster_size_table = pd.DataFrame(
    {
        "cluster_multichannel": cluster_counts.index,
        "n_samples": cluster_counts.values,
        "sample_share_percent": cluster_percentages.values,
    }
)

clustered_team_seasons.to_csv(OUTPUT_DIR / "clustered_team_seasons_multichannel.csv", index=False)
display(cluster_size_table)
print(f"Saved clustered dataset: {OUTPUT_DIR / 'clustered_team_seasons_multichannel.csv'}")


## 10. PCA Scatter Visualization

The two-dimensional PCA scatter plot is a quick visual check of separation in the first two principal components. The labeled version marks only a small number of representative team-seasons to avoid clutter.


In [ ]:
def label_for_row(row):
    parts = []
    if "team_name" in row and pd.notna(row["team_name"]):
        parts.append(str(row["team_name"]))
    if "season_name" in row and pd.notna(row["season_name"]):
        parts.append(str(row["season_name"]))
    elif "season_id" in row and pd.notna(row["season_id"]):
        parts.append(str(row["season_id"]))
    return " - ".join(parts) if parts else str(row.name)


def plot_pca_scatter(frame, output_path, label_indices=None):
    fig, ax = plt.subplots(figsize=(10, 7))
    scatter = ax.scatter(
        frame["pca_1"],
        frame["pca_2"],
        c=frame["cluster_multichannel"],
        cmap="tab10",
        alpha=0.78,
        s=45,
        edgecolor="white",
        linewidth=0.4,
    )
    ax.set_title("Experiment C - Multi-channel xT Feature Clusters")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(alpha=0.25)
    legend = ax.legend(*scatter.legend_elements(), title="Cluster", loc="best")
    ax.add_artist(legend)

    if label_indices is not None:
        for idx in label_indices:
            row = frame.loc[idx]
            ax.annotate(
                label_for_row(row),
                (row["pca_1"], row["pca_2"]),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=8,
                alpha=0.85,
            )

    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


plot_pca_scatter(clustered_team_seasons, OUTPUT_DIR / "pca_scatter_clusters_multichannel.png")

label_indices = []
for cluster_id in sorted(clustered_team_seasons["cluster_multichannel"].unique()):
    cluster_subset = clustered_team_seasons.loc[clustered_team_seasons["cluster_multichannel"] == cluster_id]
    label_indices.extend(cluster_subset.nsmallest(2, "distance_to_centroid").index.tolist())

plot_pca_scatter(
    clustered_team_seasons,
    OUTPUT_DIR / "pca_scatter_clusters_multichannel_labeled.png",
    label_indices=label_indices,
)

print(f"Saved PCA scatter plot: {OUTPUT_DIR / 'pca_scatter_clusters_multichannel.png'}")
print(f"Saved labeled PCA scatter plot: {OUTPUT_DIR / 'pca_scatter_clusters_multichannel_labeled.png'}")


## 11. Cluster Summary

This table summarizes cluster size, common competitions and teams, and any available metadata averages. Optional metadata columns are skipped gracefully when they are not present.


In [ ]:
def most_common_values(series, top_n=3):
    counts = series.dropna().astype(str).value_counts().head(top_n)
    if counts.empty:
        return ""
    return "; ".join(f"{name} ({count})" for name, count in counts.items())


summary_numeric_columns = [
    "match_count",
    "move_action_count",
    "total_positive_xT",
    "attacking_third_share",
    "middle_third_share",
    "defensive_third_share",
    "left_side_share",
    "center_share",
    "right_side_share",
    "pass_xT_share",
    "carry_xT_share",
]

cluster_summary_rows = []
for cluster_id, group in clustered_team_seasons.groupby("cluster_multichannel"):
    row = {
        "cluster_multichannel": cluster_id,
        "n_team_seasons": len(group),
        "share_of_total_samples": len(group) / len(clustered_team_seasons),
    }
    if "competition_name" in group.columns:
        row["most_common_competitions"] = most_common_values(group["competition_name"])
    if "team_name" in group.columns:
        row["most_common_teams"] = most_common_values(group["team_name"])
    for column in summary_numeric_columns:
        if column in group.columns:
            row[f"avg_{column}"] = pd.to_numeric(group[column], errors="coerce").mean()
    cluster_summary_rows.append(row)

cluster_summary = pd.DataFrame(cluster_summary_rows).sort_values("cluster_multichannel").reset_index(drop=True)
cluster_summary.to_csv(OUTPUT_DIR / "cluster_summary_multichannel.csv", index=False)
display(cluster_summary)
print(f"Saved cluster summary: {OUTPUT_DIR / 'cluster_summary_multichannel.csv'}")


## 12. Cluster Centroid Analysis

Channel-specific centroid heatmaps are the main interpretation tool for Experiment C. They show how each cluster differs in created xT, received xT, pass-created xT, carry-created xT, and action-count distribution. KMeans is fit in PCA space, but these centroids are computed in the original unscaled multi-channel feature space.


In [ ]:
cluster_centroids_multichannel = (
    clustered_team_seasons.groupby("cluster_multichannel")[feature_columns]
    .mean()
    .sort_index()
)
cluster_centroids_multichannel.to_csv(OUTPUT_DIR / "cluster_centroids_multichannel.csv")


def plot_channel_centroids(prefix, columns, output_path):
    grids = {
        cluster_id: row[columns].values.reshape(GRID_W, GRID_L)
        for cluster_id, row in cluster_centroids_multichannel.iterrows()
    }
    vmax = max(float(grid.max()) for grid in grids.values())
    vmin = 0.0
    n_clusters = len(grids)
    ncols = min(3, n_clusters)
    nrows = math.ceil(n_clusters / ncols)
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5.2 * ncols, 3.8 * nrows))
    axes = np.asarray(axes).reshape(-1)
    image = None
    for ax, (cluster_id, grid) in zip(axes, grids.items()):
        image = ax.imshow(grid, origin="lower", aspect="auto", cmap="viridis", vmin=vmin, vmax=vmax)
        ax.set_title(f"Cluster {cluster_id}")
        ax.set_xlabel("Pitch length bin")
        ax.set_ylabel("Pitch width bin")
        ax.set_xticks(range(GRID_L))
        ax.set_yticks(range(GRID_W))
        ax.tick_params(labelsize=7)
    for ax in axes[len(grids):]:
        ax.axis("off")
    fig.suptitle(f"Experiment C - {CHANNEL_DISPLAY_NAMES[prefix]} Centroids", fontsize=16, fontweight="bold")
    fig.colorbar(image, ax=axes[:len(grids)], shrink=0.78, label="Average channel share")
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


for prefix, columns in detected_channels.items():
    channel_name = CHANNEL_OUTPUT_NAMES[prefix]
    output_path = OUTPUT_DIR / f"cluster_centroids_{channel_name}_multichannel.png"
    plot_channel_centroids(prefix, columns, output_path)
    print(f"Saved {prefix} centroid heatmaps: {output_path}")

n_clusters = len(cluster_centroids_multichannel)
n_channels = len(detected_channels)
fig, axes = plt.subplots(
    nrows=n_channels,
    ncols=n_clusters,
    figsize=(3.2 * n_clusters, 2.8 * n_channels),
    squeeze=False,
)
for row_idx, (prefix, columns) in enumerate(detected_channels.items()):
    grids = {
        cluster_id: row[columns].values.reshape(GRID_W, GRID_L)
        for cluster_id, row in cluster_centroids_multichannel.iterrows()
    }
    vmax = max(float(grid.max()) for grid in grids.values())
    for col_idx, (cluster_id, grid) in enumerate(grids.items()):
        ax = axes[row_idx, col_idx]
        image = ax.imshow(grid, origin="lower", aspect="auto", cmap="viridis", vmin=0.0, vmax=vmax)
        ax.set_title(f"Cluster {cluster_id}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        if col_idx == 0:
            ax.set_ylabel(CHANNEL_DISPLAY_NAMES[prefix], fontsize=9)
fig.suptitle("Experiment C - Multi-channel Cluster Centroid Overview", fontsize=16, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "cluster_centroid_heatmaps_multichannel.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved cluster centroid vectors: {OUTPUT_DIR / 'cluster_centroids_multichannel.csv'}")
print(f"Saved combined centroid overview: {OUTPUT_DIR / 'cluster_centroid_heatmaps_multichannel.png'}")


## 13. Channel-Level Cluster Summary

This table converts every channel centroid into interpretable spatial summaries: thirds, width lanes, and the top three zones. This makes it easier to compare whether a cluster is driven by created xT, received xT, pass creation, carry creation, or action-count distribution.


In [ ]:
def zone_columns_for(prefix, columns, x_bins=None, y_bins=None):
    selected = []
    for column in columns:
        zone = zone_number_from_column(column, prefix)
        x_bin = zone % GRID_L
        y_bin = zone // GRID_L
        x_ok = x_bins is None or x_bin in x_bins
        y_ok = y_bins is None or y_bin in y_bins
        if x_ok and y_ok:
            selected.append(column)
    return selected


channel_summary_rows = []

for cluster_id, centroid in cluster_centroids_multichannel.iterrows():
    for prefix, columns in detected_channels.items():
        top_three = centroid[columns].sort_values(ascending=False).head(3)
        top_zones = [
            (zone_number_from_column(column, prefix), float(value))
            for column, value in top_three.items()
        ]
        row = {
            "cluster_multichannel": cluster_id,
            "channel_prefix": prefix,
            "channel_name": CHANNEL_DISPLAY_NAMES[prefix],
            "channel_total_mean": float(centroid[columns].sum()),
            "defensive_third_share": float(centroid[zone_columns_for(prefix, columns, x_bins=range(0, 5))].sum()),
            "middle_third_share": float(centroid[zone_columns_for(prefix, columns, x_bins=range(5, 11))].sum()),
            "attacking_third_share": float(centroid[zone_columns_for(prefix, columns, x_bins=range(11, 16))].sum()),
            "left_side_share": float(centroid[zone_columns_for(prefix, columns, y_bins=range(0, 4))].sum()),
            "center_share": float(centroid[zone_columns_for(prefix, columns, y_bins=range(4, 8))].sum()),
            "right_side_share": float(centroid[zone_columns_for(prefix, columns, y_bins=range(8, 12))].sum()),
            "top_zone_1": top_zones[0][0],
            "top_zone_1_share": top_zones[0][1],
            "top_zone_2": top_zones[1][0],
            "top_zone_2_share": top_zones[1][1],
            "top_zone_3": top_zones[2][0],
            "top_zone_3_share": top_zones[2][1],
        }
        channel_summary_rows.append(row)

cluster_channel_summary = pd.DataFrame(channel_summary_rows)
cluster_channel_summary.to_csv(OUTPUT_DIR / "cluster_channel_summary_multichannel.csv", index=False)
display(cluster_channel_summary)
print(f"Saved channel-level cluster summary: {OUTPUT_DIR / 'cluster_channel_summary_multichannel.csv'}")


## 14. Representative Team-Seasons

Representative team-seasons are the samples closest to their assigned KMeans centroid in PCA clustering space. They help manually name and interpret clusters, but they should not be treated as ground-truth labels.


In [ ]:
representative_columns = [
    "cluster_multichannel",
    "team_name",
    "season_name",
    "competition_name",
    "distance_to_centroid",
    "total_positive_xT",
    "match_count",
    "move_action_count",
]
available_representative_columns = [
    column for column in representative_columns if column in clustered_team_seasons.columns
]

representatives = (
    clustered_team_seasons.sort_values(["cluster_multichannel", "distance_to_centroid"])
    .groupby("cluster_multichannel", as_index=False)
    .head(5)
    .loc[:, available_representative_columns]
    .reset_index(drop=True)
)

representatives.to_csv(OUTPUT_DIR / "cluster_representatives_multichannel.csv", index=False)
display(representatives)
print(f"Saved representative team-seasons: {OUTPUT_DIR / 'cluster_representatives_multichannel.csv'}")


## 15. Cluster Naming Support Table

This support table gives neutral evidence for manual cluster names. Tentative descriptions are based only on observed multi-channel xT features and avoid strong labels such as counterattack, high pressing, or long-ball unless the available features directly support them.


In [ ]:
def channel_value(cluster_id, prefix, metric):
    subset = cluster_channel_summary[
        (cluster_channel_summary["cluster_multichannel"] == cluster_id)
        & (cluster_channel_summary["channel_prefix"] == prefix)
    ]
    if subset.empty or metric not in subset.columns:
        return np.nan
    return subset.iloc[0][metric]


def dominant_channel_patterns(cluster_id):
    parts = []
    for prefix in detected_channels:
        attacking = channel_value(cluster_id, prefix, "attacking_third_share")
        center = channel_value(cluster_id, prefix, "center_share")
        total = channel_value(cluster_id, prefix, "channel_total_mean")
        if pd.notna(attacking):
            parts.append(f"{CHANNEL_OUTPUT_NAMES[prefix]} attacking={attacking:.2f}")
        elif pd.notna(total):
            parts.append(f"{CHANNEL_OUTPUT_NAMES[prefix]} total={total:.2f}")
        if pd.notna(center) and center > 0.40:
            parts.append(f"{CHANNEL_OUTPUT_NAMES[prefix]} central")
    return "; ".join(parts)


def neutral_multichannel_description(cluster_id):
    created_attacking = channel_value(cluster_id, "created_z", "attacking_third_share")
    created_left = channel_value(cluster_id, "created_z", "left_side_share")
    created_center = channel_value(cluster_id, "created_z", "center_share")
    created_right = channel_value(cluster_id, "created_z", "right_side_share")
    received_attacking = channel_value(cluster_id, "received_z", "attacking_third_share")
    pass_total = channel_value(cluster_id, "pass_created_z", "channel_total_mean")
    carry_total = channel_value(cluster_id, "carry_created_z", "channel_total_mean")

    if pd.notna(created_attacking) and created_attacking > 0.45:
        return "High attacking-third created xT concentration"
    if pd.notna(created_center) and pd.notna(created_left) and pd.notna(created_right):
        if created_center > max(created_left, created_right) + 0.05:
            return "Central created xT profile"
        if max(created_left, created_right) > created_center + 0.05:
            return "Wide created xT profile"
    if pd.notna(received_attacking) and received_attacking > 0.45:
        return "High received xT in advanced zones"
    if pd.notna(pass_total) and pd.notna(carry_total):
        if pass_total > carry_total * 1.25:
            return "Pass-dominant creation profile"
        if carry_total > pass_total * 1.25:
            return "Carry-dominant creation profile"
    return "Balanced multi-channel distribution"


naming_rows = []
for cluster_id in sorted(clustered_team_seasons["cluster_multichannel"].unique()):
    row = {
        "cluster_multichannel": cluster_id,
        "n_team_seasons": int(cluster_counts.loc[cluster_id]),
        "tentative_description": neutral_multichannel_description(cluster_id),
        "dominant_channel_patterns": dominant_channel_patterns(cluster_id),
    }

    for prefix in CHANNEL_PREFIXES:
        channel_name = CHANNEL_OUTPUT_NAMES[prefix]
        if prefix in detected_channels:
            row[f"{channel_name}_top_zone_1"] = channel_value(cluster_id, prefix, "top_zone_1")
            row[f"{channel_name}_attacking_third_share"] = channel_value(cluster_id, prefix, "attacking_third_share")

    if "created_z" in detected_channels:
        row["created_left_side_share"] = channel_value(cluster_id, "created_z", "left_side_share")
        row["created_center_share"] = channel_value(cluster_id, "created_z", "center_share")
        row["created_right_side_share"] = channel_value(cluster_id, "created_z", "right_side_share")

    cluster_reps = representatives.loc[representatives["cluster_multichannel"] == cluster_id].head(3)
    if "team_name" in cluster_reps.columns:
        if "season_name" in cluster_reps.columns:
            row["example_representative_teams"] = "; ".join(
                f"{team} ({season})"
                for team, season in zip(cluster_reps["team_name"], cluster_reps["season_name"])
            )
        else:
            row["example_representative_teams"] = "; ".join(cluster_reps["team_name"].astype(str))
    else:
        row["example_representative_teams"] = ""

    naming_rows.append(row)

cluster_naming_support = pd.DataFrame(naming_rows)
cluster_naming_support.to_csv(OUTPUT_DIR / "cluster_naming_support_multichannel.csv", index=False)
display(cluster_naming_support)
print(f"Saved cluster naming support: {OUTPUT_DIR / 'cluster_naming_support_multichannel.csv'}")


## 16. Optional Comparison With Experiment A and B

If Experiment A or Experiment B clustered outputs exist, this section compares them with Experiment C. Cluster IDs are arbitrary across KMeans runs, so ARI and NMI are used for label-invariant comparison. A different grouping is not automatically worse; it may capture richer tactical distinctions.


In [ ]:
compared_with_experiment_A = False
experiment_A_overlap_n = np.nan
ARI_vs_experiment_A = np.nan
NMI_vs_experiment_A = np.nan
compared_with_experiment_B = False
experiment_B_overlap_n = np.nan
ARI_vs_experiment_B = np.nan
NMI_vs_experiment_B = np.nan


def compare_with_prior_experiment(prior_file, prior_label_column, prior_name):
    if not prior_file.exists():
        print(f"{prior_name} clustered file not found: {prior_file}. Skipping comparison.")
        return False, np.nan, np.nan, np.nan

    prior = pd.read_csv(prior_file)
    id_merge_keys = ["competition_id", "season_id", "team_id"]
    name_merge_keys = ["competition_name", "season_name", "team_name"]
    if all(column in prior.columns and column in clustered_team_seasons.columns for column in id_merge_keys):
        merge_keys = id_merge_keys
    elif all(column in prior.columns and column in clustered_team_seasons.columns for column in name_merge_keys):
        merge_keys = name_merge_keys
    else:
        print(f"{prior_name} file exists, but no stable merge keys are available. Skipping comparison.")
        return False, np.nan, np.nan, np.nan

    if prior_label_column not in prior.columns:
        print(f"{prior_name} file exists, but it does not contain {prior_label_column}. Skipping comparison.")
        return False, np.nan, np.nan, np.nan

    comparison = prior[merge_keys + [prior_label_column]].merge(
        clustered_team_seasons[merge_keys + ["cluster_multichannel"]],
        on=merge_keys,
        how="inner",
    )
    overlap_n = int(len(comparison))
    print(f"{prior_name} overlapping team-seasons: {overlap_n:,}")
    if overlap_n == 0:
        return False, overlap_n, np.nan, np.nan

    ari = adjusted_rand_score(comparison[prior_label_column], comparison["cluster_multichannel"])
    nmi = normalized_mutual_info_score(comparison[prior_label_column], comparison["cluster_multichannel"])
    slug = "experiment_A" if prior_label_column == "cluster_raw" else "experiment_B"

    overlap_counts = pd.crosstab(
        comparison[prior_label_column],
        comparison["cluster_multichannel"],
        rownames=[prior_label_column],
        colnames=["cluster_multichannel"],
    )
    overlap_row_normalized = overlap_counts.div(overlap_counts.sum(axis=1), axis=0)
    metrics = pd.DataFrame(
        [
            {
                "comparison": prior_name,
                "overlap_n": overlap_n,
                "adjusted_rand_index": ari,
                "normalized_mutual_information": nmi,
            }
        ]
    )

    metrics.to_csv(OUTPUT_DIR / f"comparison_with_{slug}_metrics.csv", index=False)
    overlap_counts.to_csv(OUTPUT_DIR / f"comparison_with_{slug}_cluster_overlap.csv")
    overlap_row_normalized.to_csv(OUTPUT_DIR / f"comparison_with_{slug}_cluster_overlap_row_normalized.csv")

    print(f"{prior_name} ARI: {ari:.4f}")
    print(f"{prior_name} NMI: {nmi:.4f}")
    display(metrics)
    display(overlap_counts)
    display(overlap_row_normalized)
    return True, overlap_n, ari, nmi


(
    compared_with_experiment_A,
    experiment_A_overlap_n,
    ARI_vs_experiment_A,
    NMI_vs_experiment_A,
) = compare_with_prior_experiment(EXPERIMENT_A_CLUSTERED_FILE, "cluster_raw", "Experiment A")

(
    compared_with_experiment_B,
    experiment_B_overlap_n,
    ARI_vs_experiment_B,
    NMI_vs_experiment_B,
) = compare_with_prior_experiment(EXPERIMENT_B_CLUSTERED_FILE, "cluster_smoothed", "Experiment B")


### Interpreting Experiment A and B Comparison

High ARI or NMI suggests the multi-channel representation preserves the main grouping structure from the simpler experiment. Low ARI or NMI suggests that multi-channel features create meaningfully different groupings. The final evaluation should consider centroid heatmap clarity, representative team-season coherence, and interpretability.


## 17. Experiment Summary

This one-row summary captures the final sample, detected channels, PCA settings, clustering scores, and optional comparisons with Experiment A and Experiment B.


In [ ]:
final_silhouette = silhouette_score(X_pca, cluster_multichannel)
final_calinski_harabasz = calinski_harabasz_score(X_pca, cluster_multichannel)
final_davies_bouldin = davies_bouldin_score(X_pca, cluster_multichannel)

experiment_summary = pd.DataFrame(
    [
        {
            "experiment_name": EXPERIMENT_NAME,
            "input_file": str(INPUT_FILE),
            "output_folder": str(OUTPUT_DIR),
            "number_of_samples": len(clustered_team_seasons),
            "number_of_features": len(feature_columns),
            "detected_channels": ", ".join(detected_channels.keys()),
            "final_k": FINAL_K,
            "pca_components_used": pca_components_used,
            "pca_explained_variance_retained": pca_explained_variance_retained,
            "final_silhouette_score": final_silhouette,
            "final_calinski_harabasz_score": final_calinski_harabasz,
            "final_davies_bouldin_score": final_davies_bouldin,
            "compared_with_experiment_A": compared_with_experiment_A,
            "experiment_A_overlap_n": experiment_A_overlap_n,
            "ARI_vs_experiment_A": ARI_vs_experiment_A,
            "NMI_vs_experiment_A": NMI_vs_experiment_A,
            "compared_with_experiment_B": compared_with_experiment_B,
            "experiment_B_overlap_n": experiment_B_overlap_n,
            "ARI_vs_experiment_B": ARI_vs_experiment_B,
            "NMI_vs_experiment_B": NMI_vs_experiment_B,
        }
    ]
)
experiment_summary.to_csv(OUTPUT_DIR / "experiment_summary_multichannel.csv", index=False)
display(experiment_summary)
print(f"Saved experiment summary: {OUTPUT_DIR / 'experiment_summary_multichannel.csv'}")


## Experiment C Interpretation Notes

Multi-channel xT clustering captures richer possession threat and ball-progression structure than a single created xT matrix. It can show whether clusters differ not only in where they create xT, but also where they receive xT, whether creation is more pass- or carry-driven, and where actions are concentrated.

The method may still miss important tactical behaviors. It does not directly measure pressing intensity, defensive block height, counterpressing, opponent quality, game state, or the full sequence context that produced the xT.

Whether adding received xT, pass-created xT, carry-created xT, and action-count distribution improves tactical interpretability should be judged from the channel-specific centroid heatmaps, the channel-level summary table, and the representative team-seasons. Channel-specific centroid heatmaps are essential because each channel carries a different meaning; averaging interpretation across channels can hide what actually separates clusters.

Clusters should be treated as hypotheses rather than final tactical labels. Experiment A is the raw single-channel baseline. Experiment B tests whether smoothing improves robustness. Experiment C tests whether multi-channel spatial features add tactical meaning. Compare all three by cluster metrics, ARI and NMI where available, centroid clarity, and whether the representative team-seasons make coherent football sense.
